# Blinded pharmacogenomics study — 110 cases, 21 genes

Genotype → diplotype → CPIC phenotype → CPIC Level A recommendation.

Model: **Claude Opus 5** (`claude-opus-5`). Sources: CPIC data API (accessed 2026-09-15) plus
the supplied guideline corpus (schema v3.1, generated 2026-05-10 from a PharmGKB bundle of
2026-05-09). Full narrative in `METHODS.md`.


In [1]:
import sys, os, csv, collections, json
sys.path.insert(0, os.getcwd())
import pgx_pipeline as P

print("cases:", len(P.cases))
print("CPIC tables cached:", {k: len(v) for k, v in P.T.items()})
print("corpus:", P.corpus["schema_version"], P.corpus["generated_at_utc"], len(P.corpus["genes"]), "genes")

cases: 110
CPIC tables cached: {'gene': 132, 'pair': 635, 'drug': 324, 'guideline': 29, 'diplotype': 111047, 'recommendation': 2115, 'allele': 1317, 'allele_definition': 1293, 'sequence_location': 1162, 'allele_location_value': 2988}
corpus: v3.1 2026-05-10T04:40:15Z 21 genes


## 1. What the cases look like

Each case gives a gene, a drug and a free-text genotype description. The descriptions mix
rsID genotypes, explicit forward-genomic annotations, structural-variant reports, repeat
lengths, HLA typing results and mitochondrial variants.


In [2]:
print(collections.Counter(c["gene"] for c in P.cases))
for c in P.cases[:3] + P.cases[3:4] + [c for c in P.cases if c["gene"] == "TPMT"][:1]:
    print(c["id"], "|", c["gene"], "|", c["drug"], "|", c["genotype"])

Counter({'CYP2D6': 20, 'CYP2C19': 20, 'CYP2C9': 9, 'DPYD': 7, 'SLCO1B1': 6, 'UGT1A1': 6, 'G6PD': 6, 'CYP2B6': 4, 'TPMT': 4, 'HLA-B*15:02': 4, 'CYP3A5': 3, 'NUDT15': 3, 'HLA-B*57:01': 2, 'HLA-A*31:01': 2, 'HLA-B*58:01': 2, 'IFNL3': 2, 'VKORC1': 2, 'CYP4F2': 2, 'CFTR': 2, 'MT-RNR1': 2, 'RYR1': 2})
70dc9c9110f3 | CYP2D6 | codeine | rs3892097 T/T (homozygous)
b07cff8c99c0 | CYP2D6 | codeine | rs3892097 C/T (heterozygous)
ce34b1a18a3d | CYP2D6 | codeine | All CYP2D6 SNPs homozygous reference
74597c5503ec | CYP2D6 | codeine | CYP2D6 *1xN duplication detected by copy-number assay (qPCR/MLPA/long-range PCR); duplications are structural variants with no defining single SNV
08e18e809fcb | TPMT | azathioprine | rs1800460 T/T + rs1142345 C/C (forward-genomic; transcript: c.460G>A + c.719A>G; TPMT on minus strand)


## 2. Stage 1 — genotype to diplotype from CPIC allele definitions

Star alleles are not hard-coded. For each interrogated position CPIC's `sequence_location`
gives the GRCh38 forward-strand HGVS, which fixes the reference and variant base; the genotype
letters in the case are then matched to that alphabet (complementing if they only fit the other
strand), and each variant base maps to the allele CPIC defines minimally by it.


In [3]:
for gene, rs in [("CYP2D6","rs3892097"), ("CYP2C19","rs4244285"), ("CYP2C19","rs12248560"),
                 ("TPMT","rs1800460"), ("TPMT","rs1142345"), ("CYP3A5","rs776746"),
                 ("VKORC1","rs9923231"), ("G6PD","rs5030868"), ("DPYD","rs56038477")]:
    s = P.locations_for(gene, rs)[0]
    ref = P.LOC[s["id"]]["ref"]
    alts = sorted(b for b in P.observed_bases(gene, rs) if b != ref and len(b) == 1)
    mins = {b: P.minimal_allele(gene, rs, b)[0] for b in alts}
    print(f"{gene:8s} {rs:12s} {s['chromosomelocation']:22s} ref={ref}  variant->allele {mins}")

CYP2D6   rs3892097    g.42128945C>T          ref=C  variant->allele {'T': '*4'}
CYP2C19  rs4244285    g.94781859G>A          ref=G  variant->allele {'A': '*2'}
CYP2C19  rs12248560   g.94761900C>T          ref=C  variant->allele {'T': '*17', 'Y': '*4'}
TPMT     rs1800460    g.18138997C>T          ref=C  variant->allele {'T': '*3B'}
TPMT     rs1142345    g.18130687T>C; g.18130687T>G ref=T  variant->allele {'C': '*3C', 'G': '*41'}
CYP3A5   rs776746     g.99672916T>C          ref=T  variant->allele {'C': '*3'}
VKORC1   rs9923231    g.31096368C>T          ref=C  variant->allele {'T': 'rs9923231 variant (T)'}
G6PD     rs5030868    g.154534419G>A         ref=G  variant->allele {'A': 'Mediterranean, Dallas, Panama, Sassari, Cagliari, Birmingham'}
DPYD     rs56038477   g.97573863C>T          ref=C  variant->allele {'T': 'c.1129-5923C>G, c.1236G>A (HapB3)'}


In [4]:
# the strand traps: which cases had to be complemented onto the forward strand
for c in P.cases:
    r = P.resolve_genotype(c)
    if r["orientation"] == "complemented to forward strand":
        print(f"{c['id']} {c['gene']:8s} {c['genotype'][:48]:50s} -> {r['diplotype']}  ({r['hgvs_g']})")

743cd9ebf0e9 CYP3A5   rs776746 A/A (homozygous *1)                       -> *1/*1  (g.99672916T>C)
22c4e14a3cf9 CYP3A5   rs776746 A/G (heterozygous)                        -> *1/*3  (g.99672916T>C)
de1d2a9b4bed CYP3A5   rs776746 G/G (homozygous)                          -> *3/*3  (g.99672916T>C)
c1e4c84b49dd VKORC1   rs9923231 A/A (homozygous, -1639 promoter)         -> rs9923231 variant (T)/rs9923231 variant (T)  (g.31096368C>T)
a83853be69dc VKORC1   rs9923231 G/G (homozygous reference)               -> rs9923231 reference (C)/rs9923231 reference (C)  (g.31096368C>T)


## 3. Stage 2 and 3 — phenotype and recommendation

Phenotype comes from CPIC's `diplotype` table (111k rows for these genes); recommendation from
CPIC's `recommendation` table keyed on the gene lookup value. 95 of 110 cases resolve to exactly
one CPIC recommendation row and none to more than one. The 15 that resolve to none fall outside
CPIC's machine-readable recommendations (warfarin is an algorithm, ondansetron's table is not
populated in the API, irinotecan has no CPIC guideline, IFNL3–peginterferon is retired) and are
taken from the supplied corpus with the source named.


In [5]:
nhits = collections.Counter()
fallbacks = []
for c in P.cases:
    r = P.resolve_genotype(c)
    phen, lk, src, _ = P.phenotype_for(r["gene"], r["diplotype"], c["genotype"])
    hits, dn = P.recommendation_for(c["drug"], r["gene"], lk)
    nhits[len(hits)] += 1
    if not hits:
        fallbacks.append((c["drug"], r["gene"]))
print("CPIC recommendation rows matched per case:", dict(nhits))
print("corpus-sourced:", collections.Counter(fallbacks))
print()
for p in P.T["pair"]:
    if p["genesymbol"] == "IFNL3" and p["drugid"] == P.DRUG_ID["peginterferon alfa-2a"]:
        print("IFNL3 / peginterferon alfa-2a cpiclevel:", p["cpiclevel"])
for p in P.T["pair"]:
    if p["genesymbol"] == "UGT1A1" and p["drugid"] == P.DRUG_ID["irinotecan"]:
        print("UGT1A1 / irinotecan cpiclevel:", p["cpiclevel"], "guidelineid:", p["guidelineid"],
              "pgxtesting:", p["pgxtesting"])

CPIC recommendation rows matched per case: {1: 95, 0: 15}
corpus-sourced: Counter({('ondansetron', 'CYP2D6'): 3, ('warfarin', 'CYP2C9'): 3, ('irinotecan', 'UGT1A1'): 3, ('PEG-IFN-α', 'IFNL3'): 2, ('warfarin', 'VKORC1'): 2, ('warfarin', 'CYP4F2'): 2})

IFNL3 / peginterferon alfa-2a cpiclevel: Retired
UGT1A1 / irinotecan cpiclevel: A guidelineid: None pgxtesting: Testing Recommended


## 4. Run the pipeline and write the deliverables

In [6]:
answers, audit = P.main()

cases=110 answered=110 abstained=0 lethal_class=22
unique diplotype->phenotype rows=60  phenotype->recommendation rows=106


## 5. Results

In [7]:
print(collections.Counter(a["phenotype"] for a in audit).most_common(8))
print()
print("abstained:", sum(1 for a in answers if a["abstained"] == "true"))
print("with a recorded assumption:", sum(1 for a in audit if a["assumptions"]))
print("with a recorded ambiguity:", sum(1 for a in audit if a["ambiguity"]))
print("mapping methods:", dict(collections.Counter(a["mapping_method"] for a in audit)))

[('Intermediate Metabolizer', 22), ('Poor Metabolizer', 20), ('Normal Metabolizer', 20), ('Ultrarapid Metabolizer', 9), ('Rapid Metabolizer', 5), ('Deficient', 3), ('Poor Function', 2), ('Decreased Function', 2)]

abstained: 0
with a recorded assumption: 86
with a recorded ambiguity: 18
mapping methods: {'CPIC allele-definition lookup (single SNV)': 59, 'reference-statement': 16, 'structural-variant handler': 5, 'two-SNV phasing handler': 1, 'two-SNV haplotype handler': 1, 'repeat-length handler': 6, 'HLA tag-SNP handler': 2, 'HLA typing handler': 8, 'G6PD haplotype handler': 6, 'CFTR handler': 2, 'mitochondrial variant handler': 2, 'RYR1 handler': 2}


In [8]:
print("lethal-class cases:", sum(1 for a in audit if a["lethal_class"] == "true"))
for a in audit:
    if a["lethal_class"] == "true":
        print(f"  {a['case_id']}  {a['gene']:12s} {a['drug']:22s} {a['phenotype'][:40]:42s} {a['lethal_rationale'][:60]}")

lethal-class cases: 22
  74597c5503ec  CYP2D6       codeine                Ultrarapid Metabolizer                     Ultrarapid morphine formation from codeine has caused fatal 
  ca9e14feca53  CYP2C19      clopidogrel            Poor Metabolizer                           Loss of clopidogrel activation causes stent thrombosis and m
  97c3baebed4e  CYP2C9       warfarin               Poor Metabolizer                           Standard warfarin dosing in a CYP2C9 poor metaboliser causes
  e85ecfa887cb  CYP2C9       phenytoin              Poor Metabolizer                           Standard phenytoin dosing causes toxic concentrations and en
  b94a43e574db  DPYD         fluorouracil           Poor Metabolizer                           Complete DPD deficiency; full-dose fluoropyrimidine causes f
  c57caa44a38a  DPYD         fluorouracil           Intermediate Metabolizer                   Partial DPD deficiency; full-dose fluoropyrimidine causes gr
  6b8d2db91001  DPYD         fluorouracil

In [9]:
# the cases closest to the abstention threshold, with their recorded ambiguity
for a in audit:
    if a["mapping_method"] == "structural-variant handler":
        print(a["case_id"], a["drug"], "->", a["diplotype"], "|", a["phenotype"])
        print("   assumptions:", a["assumptions"])
        print("   ambiguity  :", a["ambiguity"], "\n")

74597c5503ec codeine -> *1/*1x2 | Ultrarapid Metabolizer
   assumptions: N assumed to be 2 (*1x2); the assay reports *1xN without resolving copy number. | The non-duplicated chromosome is assumed to be *1, as no SNV-defined variant was reported.
   ambiguity  : If N>2 the activity score is higher but the phenotype stays Ultrarapid. If the second chromosome actually carried a no-function allele (e.g. *4/*1x2, activity score 2.0) the phenotype would be Normal Metabolizer - copy-number assays alone cannot phase the duplication to an allele. 

61741d10f238 tamoxifen -> *1/*1x2 | Ultrarapid Metabolizer
   assumptions: N assumed to be 2 (*1x2); the assay reports *1xN without resolving copy number. | The non-duplicated chromosome is assumed to be *1, as no SNV-defined variant was reported.
   ambiguity  : If N>2 the activity score is higher but the phenotype stays Ultrarapid. If the second chromosome actually carried a no-function allele (e.g. *4/*1x2, activity score 2.0) the phenotype would 

## 6. Deliverables written to this folder

- `claude_science_answers.csv` — 110 answers
- `table_diplotype_to_phenotype.csv`, `table_phenotype_to_recommendation.csv` — the lookup tables built
- `case_audit.csv` — per-case provenance, assumptions, ambiguities, lethal flag
- `source_versions.csv` — per-gene source and version record
- `METHODS.md` — approach, sources, lethal-class rationale, error estimate
